# 01: Initial Data Pipeline, Frame Extraction & Dataset Loader
**Group 11 · COSE474 Deep Learning · Korea University · Spring 2026**


This notebook does two things:
1. Extracts 16 evenly-spaced frames from each raw `.MP4` video and saves them as JPGs into `data/frames/`
2. Defines the `KSLDataset` PyTorch class used by every other notebook in this project

**Dataset:** [KSL-77 (Yangseung/KSL)](https://github.com/Yangseung/KSL) — 1,540 videos across 77 Korean sign classes, 20 signers, recorded at 30fps.

Raw video structure coming in:
```
data/raw/
  01/   ← class 1 (sign word #1)
    00_01.MP4   ← signer 00, class 01
    01_01.MP4
    ...
  02/
  ...
  77/
```

Output structure going out:
```
data/frames/
  00_01/   ← 16 JPGs from signer 00, class 01
    frame_000.jpg
    ...
    frame_015.jpg
  01_01/
  ...
```

# Mount Drive + config

In [1]:
# Author: Marcello Nico Valerian
# Notebook 01 — data pipeline + dataset loader

from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/KSL_DL2026')
import config

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Imports

In [2]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

random.seed(config.RANDOM_SEED)
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)

# Frame Extraction
Each video is ~120 frames at 30fps (\~4 seconds). We sample `NUM_FRAMES = 16` frames evenly across the clip, same setting used by the original KSL paper's LRCN baseline. Frames are resized to 224×224 to match ImageNet pretrained backbone input.

The extraction skips clips that already have all 16 frames saved, so re-running this cell is safe.

In [3]:
def extract_frames(video_path, output_dir, num_frames=config.NUM_FRAMES, img_size=config.IMG_SIZE):
    """
    Sample `num_frames` evenly-spaced frames from a video and save as JPGs.
    Uses np.linspace to pick indices (covers start and end of the sign).
    Returns the number of frames actually saved.
    """
    os.makedirs(output_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total == 0:
        cap.release()
        return 0

    # If the clip is shorter than what we want, take every frame we have
    if total < num_frames:
        indices = list(range(total))
    else:
        indices = np.linspace(0, total - 1, num_frames, dtype=int)

    saved = 0
    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.resize(frame, (img_size, img_size))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # cv2 reads BGR by default
        Image.fromarray(frame).save(os.path.join(output_dir, f'frame_{i:03d}.jpg'))
        saved += 1

    cap.release()
    return saved

## Run extraction over all 1540 clips

In [4]:
raw_dir = config.DATA_RAW      # .../data/raw/
out_dir = config.DATA_FRAMES   # .../data/frames/

done, skipped, failed = 0, 0, []

for class_folder in sorted(os.listdir(raw_dir)):           # '01' .. '77'
    class_path = os.path.join(raw_dir, class_folder)
    if not os.path.isdir(class_path):
        continue

    for video_file in sorted(os.listdir(class_path)):      # 'XX_YY.MP4'
        if not video_file.upper().endswith('.MP4'):
            continue

        stem = os.path.splitext(video_file)[0]             # e.g. '00_01'
        output_clip_dir = os.path.join(out_dir, stem)

        # Skip if already extracted (all frames present)
        if os.path.exists(output_clip_dir):
            existing = os.listdir(output_clip_dir)
            if len(existing) == config.NUM_FRAMES:
                skipped += 1
                continue

        n = extract_frames(os.path.join(class_path, video_file), output_clip_dir)

        if n == 0:
            failed.append(video_file)
        else:
            done += 1

print(f"Extracted : {done} clips")
print(f"Skipped   : {skipped} clips (already done)")
print(f"Failed    : {len(failed)}")
if failed:
    print("Failed files:", failed)

Extracted : 1229 clips
Skipped   : 0 clips (already done)
Failed    : 0


# Dataset Class

`KSLDataset` wraps the extracted frames folder for use with PyTorch's `DataLoader`.

Folder name is `{signer}_{class}` (e.g. `04_12` = signer 04, class 12). Class is 1-indexed in the folder name, so we subtract 1 to get a 0-indexed label for PyTorch.

```
folder '00_01' → class_id = 1 → label = 0
folder '19_77' → class_id = 77 → label = 76
```

In [5]:
class KSLDataset(Dataset):
    def __init__(self, clip_dirs, transform=None):
        """
        Args:
            clip_dirs : list of absolute paths to clip folders (each has 16 JPGs)
            transform : torchvision transform applied to each frame individually
        """
        self.clips = clip_dirs
        self.transform = transform

    def __len__(self):
        return len(self.clips)

    def __getitem__(self, idx):
        clip_path = self.clips[idx]
        folder_name = os.path.basename(clip_path)       # e.g. '04_12'
        class_id = int(folder_name.split('_')[1])       # 12
        label = class_id - 1                            # 11 (0-indexed)

        frame_files = sorted(os.listdir(clip_path))

        # Pad with last frame if somehow short, truncate if long
        if len(frame_files) < config.NUM_FRAMES:
            frame_files += [frame_files[-1]] * (config.NUM_FRAMES - len(frame_files))
        frame_files = frame_files[:config.NUM_FRAMES]

        tensors = []
        for fname in frame_files:
            img = Image.open(os.path.join(clip_path, fname)).convert('RGB')
            if self.transform:
                img = self.transform(img)
            tensors.append(img)

        # Stack → (NUM_FRAMES, C, H, W)
        return torch.stack(tensors), label

# Sanity Check

Quick end-to-end test before handing off to notebook 02. We're not splitting train/val here, that's done with 5-fold cross-validation in the individual model notebooks (the split is by signer, not by video, to avoid data leakage).

In [6]:
# Transforms + sanity check
transform = transforms.Compose([
    transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=config.NORMALIZE_MEAN, std=config.NORMALIZE_STD),
])

# Collect all extracted clip folders
all_clips = sorted([
    os.path.join(config.DATA_FRAMES, d)
    for d in os.listdir(config.DATA_FRAMES)
    if os.path.isdir(os.path.join(config.DATA_FRAMES, d))
])

print(f"Total clips found: {len(all_clips)}")   # should be ~1540

# Spot-check the dataset with a tiny loader
sample_ds = KSLDataset(all_clips[:8], transform=transform)
sample_dl = DataLoader(sample_ds, batch_size=2, shuffle=False)

frames_batch, labels_batch = next(iter(sample_dl))

print(f"Frames tensor shape : {frames_batch.shape}")  # expect (2, 16, 3, 224, 224)
print(f"Labels              : {labels_batch}")
print(f"Label range         : {labels_batch.min().item()} – {labels_batch.max().item()}")

# Check pixel range after normalization (roughly -2.1 to 2.6 for ImageNet norm)
print(f"Pixel min/max       : {frames_batch.min():.2f} / {frames_batch.max():.2f}")

Total clips found: 1229
Frames tensor shape : torch.Size([2, 16, 3, 224, 224])
Labels              : tensor([0, 1])
Label range         : 0 – 1
Pixel min/max       : -2.12 / 2.64


Frames are in `data/frames/`. The `KSLDataset` class above is the one to use in all subsequent notebooks.